# EoS ablation on LLaDA — Colab (A100)

方向 1 第一步：EoS 带来的提升来自 token 本身还是额外位置，以及靠哪条注意力通路起作用。代码说明见仓库 README。

**运行前**：Runtime → Change runtime type → **A100 GPU**。

结果写到 Google Drive，Colab 断线后重新运行全部单元格即可，已完成的条件会自动跳过。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 挂载 Drive，设置输出目录

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/eos_ablation/results_step1'   # 结果保存在 Drive，断线不丢
MODEL = 'GSAI-ML/LLaDA-1.5'          # 可换成 'GSAI-ML/LLaDA-8B-Instruct'
PER_LEVEL = 40                       # 每个难度取多少题；0 = 全量
BATCH = 16                           # A100 40GB 用 16；80GB 可以用 32
import os; os.makedirs(OUT, exist_ok=True)

## 2. 代码和数据

如果 EOS-Token 是私有仓库：在左侧 🔑 Secrets 里添加 `GITHUB_TOKEN`（一个有 repo 读权限的 personal access token），并打开 notebook access。公开仓库则不需要。

In [ ]:
import os, subprocess
url = 'https://github.com/GEOFFREY0827/EOS-Token.git'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    url = f'https://{tok}@github.com/GEOFFREY0827/EOS-Token.git'
except Exception:
    pass
%cd /content
if os.path.exists('EOS-Token'):
    !git -C EOS-Token pull -q
else:
    subprocess.run(['git', 'clone', '-q', url, 'EOS-Token'], check=True)
%cd /content/EOS-Token
if not os.path.exists('thinking_eos-by-eos'):
    !git clone -q https://github.com/minaolensarah/thinking_eos-by-eos.git
!ls

## 3. 依赖

LLaDA 的 remote code 对 transformers 版本敏感，这里固定为 Breckner & Schuster 使用的 4.57.3。如果这个单元格之后 import 报错，选 Runtime → Restart session，然后从第 1 步重新运行（不需要重装）。

In [ ]:
!pip install -q transformers==4.57.3 accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.get_device_name(0))

## 4. Self-test（必须 PASS）

检查注入的 attention mask 是否真的生效。第一次运行会下载约 16GB 的模型权重，需要几分钟。

显示 FAIL 时，可以把 `--dtype bf16` 改成 `--dtype fp32` 再跑一次，排除 bf16 数值噪声；如果仍然 FAIL，attention 消融的结果就不可信。

In [ ]:
!python eos_ablation.py --model $MODEL --data_dir thinking_eos-by-eos/datasets --task maths --selftest --dtype bf16

## 5. 实验

每个任务两组：
- **填充物 × 尾部长度**：`eos` vs `mask`，n = 0, 8, 32, 64, 128
- **注意力通路消融**：n = 64，`noread / isolated / nowrite_A / nowrite_P / nowrite_all`

建议先只跑 maths（最快，约十几分钟），确认数字合理后再跑 boxes 和 sudoku。

In [ ]:
def run(task, n_tail, fillers, attn):
    !python eos_ablation.py --model $MODEL --data_dir thinking_eos-by-eos/datasets --out_dir $OUT --per_level $PER_LEVEL --batch_size $BATCH --task $task --n_tail $n_tail --fillers $fillers --attn $attn

run('maths', '0,8,32,64,128', 'eos,mask', 'full')
run('maths', '64', 'eos,mask', 'noread,isolated,nowrite_A,nowrite_P,nowrite_all')

In [ ]:
!python summarize.py --out_dir $OUT

确认 maths 的结果合理后，再跑另外两个任务：

In [ ]:
for task in ['boxes', 'sudoku']:
    run(task, '0,8,32,64,128', 'eos,mask', 'full')
    run(task, '64', 'eos,mask', 'noread,isolated,nowrite_A,nowrite_P,nowrite_all')

## 6. 汇总

In [ ]:
!python summarize.py --out_dir $OUT
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUT}/*_filler_vs_tail.png')):
    print(p); display(Image(p))